# FINN build Miguel - Red 16x12 RGB

Notebook preparado para tu red cuantizada con:

- Entrada: `1 x 3 x 16 x 12`
- Total de entrada: `576` valores
- Arquitectura: `576 -> 256 -> 128 -> 16 -> 2`
- Salida: `2` valores
- FPGA: `xc7z020clg400-1`
- Folding manual para evitar el error de Vitis HLS por buses de mas de `8191` bits.

Ejecuta las celdas de arriba hacia abajo. Si el kernel entra en `ipdb>`, escribe `q` y pulsa Enter.

In [1]:
# 1. Imports generales
import os
import json
import shutil
from pathlib import Path

import torch
import torch.nn as nn
import brevitas.nn as qnn
import brevitas.onnx as bo

print("Imports cargados correctamente")

Imports cargados correctamente


In [2]:
# 2. Configuracion principal

WEIGHTS_FILE = "state_dict_quantized_last.pth"
MODEL_ONNX = "model.onnx"
MODEL_TIDY_ONNX = "model_tidy.onnx"
FINAL_OUTPUT_DIR = "output_final_miguel"
FOLDING_FILE = "folding_config_miguel.json"

# Tu red real usa imagen RGB 16x12: 3 * 16 * 12 = 576 valores
INPUT_SHAPE = (1, 3, 16, 12)

# FPGA vista en el wrapper/stub de Vivado
FPGA_PART = "xc7z020clg400-1"

print("Carpeta actual:", os.getcwd())
print("INPUT_SHAPE:", INPUT_SHAPE)
print("Pesos esperados:", WEIGHTS_FILE)
print("Existe fichero de pesos:", os.path.exists(WEIGHTS_FILE))

Carpeta actual: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example
INPUT_SHAPE: (1, 3, 16, 12)
Pesos esperados: state_dict_quantized_last.pth
Existe fichero de pesos: True


In [3]:
# 3. Definicion de la red RedMiguel

from brevitas.quant import Int8Bias

class RedMiguel(nn.Module):
    def __init__(self, in_width=8, weight_width=4, act_width=4, out_width=8):
        super().__init__()

        self.quant_input = qnn.QuantIdentity(return_quant_tensor=True, bit_width=in_width)

        self.fc1 = qnn.QuantLinear(
            in_features=576, out_features=256,
            weight_bit_width=weight_width, bias=True,
            bias_quant=Int8Bias, return_quant_tensor=True
        )
        self.relu1 = qnn.QuantReLU(bit_width=act_width, return_quant_tensor=True)

        self.fc2 = qnn.QuantLinear(
            in_features=256, out_features=128,
            weight_bit_width=weight_width, bias=True,
            bias_quant=Int8Bias, return_quant_tensor=True
        )
        self.relu2 = qnn.QuantReLU(bit_width=act_width, return_quant_tensor=True)

        self.fx = qnn.QuantLinear(
            in_features=128, out_features=16,
            weight_bit_width=weight_width, bias=True,
            bias_quant=Int8Bias, return_quant_tensor=True
        )
        self.relu3 = qnn.QuantReLU(bit_width=act_width, return_quant_tensor=True)

        self.predictor = qnn.QuantLinear(
            in_features=16, out_features=2,
            weight_bit_width=weight_width, bias=True,
            bias_quant=Int8Bias, return_quant_tensor=True
        )

        self.quant_output = qnn.QuantIdentity(return_quant_tensor=True, bit_width=out_width)

    def forward(self, x):
        x = self.quant_input(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fx(x)
        x = self.relu3(x)
        x = self.predictor(x)
        x = self.quant_output(x)
        return x

model = RedMiguel(in_width=8, weight_width=4, act_width=4, out_width=8)
model.eval()

print(model)

RedMiguel(
  (quant_input): QuantIdentity(
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (act_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (fused_activation_quant_proxy): FusedActivationQuantProxy(
        (activation_impl): Identity()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClamp()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): ParameterFromRuntimeStatsScaling(
            (stats_input_view_shape_impl): OverTensorView()
            (stats): _Stats(
              (stats_impl): AbsPercentile()
            )
            (restrict_clamp_scaling): _RestrictClampValue(
              (clamp_min_ste): Identity()
              (restrict_value_impl): FloatRestrictValue()
            )
            (

In [4]:
# 4. Carga de pesos

assert os.path.exists(WEIGHTS_FILE), f"No encuentro {WEIGHTS_FILE}. Ponlo en la misma carpeta que este notebook."

state_dict = torch.load(WEIGHTS_FILE, map_location="cpu")

if isinstance(state_dict, dict) and "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

clean_state_dict = {}
for k, v in state_dict.items():
    new_k = k
    if new_k.startswith("module."):
        new_k = new_k[len("module."):]
    if new_k.startswith("model."):
        new_k = new_k[len("model."):]
    clean_state_dict[new_k] = v

missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)
model.eval()

print("Pesos cargados desde:", WEIGHTS_FILE)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

if len(missing) == 0 and len(unexpected) == 0:
    print("Carga perfecta: todas las capas coinciden.")
else:
    print("Ojo: revisa missing/unexpected. Si salen listas vacias, esta bien.")

Pesos cargados desde: state_dict_quantized_last.pth
Missing keys: []
Unexpected keys: []
Carga perfecta: todas las capas coinciden.


In [5]:
# 5. Prueba rapida de entrada y salida

with torch.no_grad():
    dummy_input = torch.randn(*INPUT_SHAPE)
    dummy_output = model(dummy_input)

print("Forma entrada:", tuple(dummy_input.shape))
print("Numero valores entrada:", dummy_input.numel())
print("Forma salida:", tuple(dummy_output.shape))
print("Salida ejemplo:", dummy_output)

assert dummy_input.numel() == 576, "La entrada debe tener 576 valores: 3*16*12"
assert tuple(dummy_output.shape) == (1, 2), "La salida esperada es (1, 2)"

Forma entrada: (1, 3, 16, 12)
Numero valores entrada: 576
Forma salida: (1, 2)
Salida ejemplo: QuantTensor(value=tensor([[-0.1533,  0.1666]]), scale=tensor(0.0067), zero_point=tensor(0.), bit_width=tensor(8.), signed_t=tensor(True), training_t=tensor(False))


In [6]:
# 6. Exportacion a ONNX compatible con FINN

if os.path.exists(MODEL_ONNX):
    os.remove(MODEL_ONNX)

bo.export_finn_onnx(model, INPUT_SHAPE, MODEL_ONNX)

print("Modelo exportado a:", MODEL_ONNX)
print("Existe:", os.path.exists(MODEL_ONNX))

Modelo exportado a: model.onnx
Existe: True


In [7]:
# 7. Comprobacion del ONNX exportado

from qonnx.core.modelwrapper import ModelWrapper

model_check = ModelWrapper(MODEL_ONNX)

print("Entradas:")
for inp in model_check.graph.input:
    print(" ", inp.name, model_check.get_tensor_shape(inp.name))

print("\nSalidas:")
for out in model_check.graph.output:
    print(" ", out.name, model_check.get_tensor_shape(out.name))

Entradas:
  inp.1 [1, 3, 16, 12]
  86 [1]

Salidas:
  85 [1, 2]


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/core/modelwrapper.py:93: UserWarning: Some old-style domain attributes were automatically converted to new-style,
                i.e. domain=finn to domain=qonnx.custom_op.<general|fpgadataflow|...>
  warnings.warn(


In [8]:
# 8. Tidy basico del modelo ONNX
# Si alguna transformacion no esta disponible en tu version, el build_dataflow tambien hara limpieza interna.

from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants

model_tidy = ModelWrapper(MODEL_ONNX)

for trn in [
    InferShapes(),
    FoldConstants(),
    GiveUniqueNodeNames(),
    GiveReadableTensorNames(),
    InferDataTypes(),
    RemoveStaticGraphInputs(),
]:
    model_tidy = model_tidy.transform(trn)

model_tidy.save(MODEL_TIDY_ONNX)

print("Modelo tidy guardado en:", MODEL_TIDY_ONNX)
print("Existe:", os.path.exists(MODEL_TIDY_ONNX))

Modelo tidy guardado en: model_tidy.onnx
Existe: True


In [9]:
# 9. Crear folding manual para evitar el error de bitwidth > 8191
# Error anterior: PE=256, SIMD=9, peso=4 bits -> 256*9*4 = 9216 bits.
# Nuevo: PE=16, SIMD=8 -> 16*8*4 = 512 bits.

folding_config = {
    "Defaults": {},
    "MatrixVectorActivation_0": {"PE": 16, "SIMD": 8, "ram_style": "auto", "resType": "auto"},
    "MatrixVectorActivation_1": {"PE": 16, "SIMD": 8, "ram_style": "auto", "resType": "auto"},
    "MatrixVectorActivation_2": {"PE": 8, "SIMD": 8, "ram_style": "auto", "resType": "auto"},
    "MatrixVectorActivation_3": {"PE": 1, "SIMD": 8, "ram_style": "auto", "resType": "auto"}
}

with open(FOLDING_FILE, "w") as f:
    json.dump(folding_config, f, indent=4)

print("Folding guardado en:", FOLDING_FILE)
print(json.dumps(folding_config, indent=4))

Folding guardado en: folding_config_miguel.json
{
    "Defaults": {},
    "MatrixVectorActivation_0": {
        "PE": 16,
        "SIMD": 8,
        "ram_style": "auto",
        "resType": "auto"
    },
    "MatrixVectorActivation_1": {
        "PE": 16,
        "SIMD": 8,
        "ram_style": "auto",
        "resType": "auto"
    },
    "MatrixVectorActivation_2": {
        "PE": 8,
        "SIMD": 8,
        "ram_style": "auto",
        "resType": "auto"
    },
    "MatrixVectorActivation_3": {
        "PE": 1,
        "SIMD": 8,
        "ram_style": "auto",
        "resType": "auto"
    }
}


In [10]:
# 10. Configuracion de build_dataflow

from finn.builder import build_dataflow as build
from finn.builder import build_dataflow_config as build_cfg

if os.path.exists(FINAL_OUTPUT_DIR):
    shutil.rmtree(FINAL_OUTPUT_DIR)
    print("Salida anterior eliminada:", FINAL_OUTPUT_DIR)

cfg = build.DataflowBuildConfig(
    output_dir=FINAL_OUTPUT_DIR,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
    ],
    folding_config_file=FOLDING_FILE,
    auto_fifo_depths=False,
    mvau_wwidth_max=36,
    target_fps=1000,
    synth_clk_period_ns=10.0,
    fpga_part=FPGA_PART
)

print("Configuracion creada")
print("output_dir:", FINAL_OUTPUT_DIR)
print("folding_config_file:", FOLDING_FILE)
print("fpga_part:", FPGA_PART)

Salida anterior eliminada: output_final_miguel
Configuracion creada
output_dir: output_final_miguel
folding_config_file: folding_config_miguel.json
fpga_part: xc7z020clg400-1


In [11]:
import os

os.environ["FINN_BUILD_DIR"] = "/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build"

os.makedirs(os.environ["FINN_BUILD_DIR"], exist_ok=True)

print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])

FINN_BUILD_DIR = /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build


In [12]:
import os

os.environ["VIVADO_PATH"] = "/tools/Xilinx/Vivado/2022.2"
os.environ["XILINX_VIVADO"] = "/tools/Xilinx/Vivado/2022.2"

print("VIVADO_PATH =", os.environ["VIVADO_PATH"])
print("Existe Vivado:", os.path.exists(os.environ["VIVADO_PATH"]))

VIVADO_PATH = /tools/Xilinx/Vivado/2022.2
Existe Vivado: True


In [13]:
# 11. Ejecutar build_dataflow
# Esta celda puede tardar mucho. Si aparece ipdb>, escribe q y pulsa Enter.

build.build_dataflow_cfg(MODEL_TIDY_ONNX, cfg)

print("Build finalizado. Revisa la carpeta:", FINAL_OUTPUT_DIR)

Building dataflow accelerator from model_tidy.onnx
Intermediate outputs will be generated in /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build
Final outputs will be generated in output_final_miguel
Build log is at output_final_miguel/build_dataflow.log
Running step: step_qonnx_to_finn [1/17]
Running step: step_tidy_up [2/17]
Running step: step_streamline [3/17]
Running step: step_convert_to_hls [4/17]
Running step: step_create_dataflow_partition [5/17]
Running step: step_target_fps_parallelization [6/17]
Running step: step_apply_folding_config [7/17]
Running step: step_generate_estimate_reports [8/17]
Running step: step_hls_codegen [9/17]
Running step: step_hls_ipgen [10/17]
Running step: step_set_fifo_depths [11/17]
Running step: step_create_stitched_ip [12/17]
Running step: step_measure_rtlsim_performance [13/17]
Running step: step_out_of_context_synthesis [14/17]
Running step: step_synthesize_bitfile [15/17]
Running step: step_make_pynq_driver [16/17]
Running step: 

In [14]:
# 12. Comprobacion rapida de resultados generados

print("Existe salida final:", os.path.exists(FINAL_OUTPUT_DIR))

if os.path.exists(FINAL_OUTPUT_DIR):
    print("Contenido principal:")
    for x in os.listdir(FINAL_OUTPUT_DIR):
        print(" -", x)

    log_path = os.path.join(FINAL_OUTPUT_DIR, "build_dataflow.log")
    print("\nExiste build_dataflow.log:", os.path.exists(log_path))
    if os.path.exists(log_path):
        with open(log_path, "r", errors="ignore") as f:
            log = f.read()
        print("Contiene Traceback:", "Traceback" in log)
        print("Ultimas lineas del log:")
        print(log[-3000:])

Existe salida final: True
Contenido principal:
 - build_dataflow.log
 - report
 - stitched_ip
 - time_per_step.json
 - auto_folding_config.json
 - final_hw_config.json
 - intermediate_models

Existe build_dataflow.log: True
Contiene Traceback: False
Ultimas lineas del log:
1/bd/finn_design/ip/finn_design_MatrixVectorActivation_0_0/finn_design_MatrixVectorActivation_0_0.xci -> output_final_miguel/stitched_ip/finn_vivado_stitch_proj.srcs/sources_1/bd/finn_design/ip/finn_design_MatrixVectorActivation_0_0
[2026-06-16 08:34:14,063] creating output_final_miguel/stitched_ip/finn_vivado_stitch_proj.srcs/sources_1/bd/finn_design/ip/finn_design_MatrixVectorActivation_3_wstrm_0
[2026-06-16 08:34:14,063] copying /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/vivado_stitch_proj_0598zxq9/finn_vivado_stitch_proj.srcs/sources_1/bd/finn_design/ip/finn_design_MatrixVectorActivation_3_wstrm_0/finn_design_MatrixVectorActivation_3_wstrm_0.xci -> output_final_miguel/stitched_ip/finn_viva

In [15]:
# 13. Buscar archivos importantes para Verilator/Vivado

if os.path.exists(FINAL_OUTPUT_DIR):
    found = []
    for root, dirs, files in os.walk(FINAL_OUTPUT_DIR):
        for f in files:
            if ("all_verilog_srcs" in f or "finn_design_wrapper" in f or f.endswith(".v") or f.endswith(".sv") or f.endswith(".dcp") or f.endswith(".xo")):
                found.append(os.path.join(root, f))

    print("Archivos importantes encontrados:", len(found))
    for p in found[:200]:
        print(p)

    if len(found) == 0:
        print("No se han encontrado Verilog/DCP. El build seguramente no llego a STITCHED_IP.")

Archivos importantes encontrados: 121
output_final_miguel/stitched_ip/all_verilog_srcs.txt
output_final_miguel/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ip/finn_design_StreamingFIFO_0_0/sim/finn_design_StreamingFIFO_0_0.v
output_final_miguel/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ip/finn_design_StreamingFIFO_0_0/synth/finn_design_StreamingFIFO_0_0.v
output_final_miguel/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ip/finn_design_dwc_1/sim/finn_design_dwc_1.v
output_final_miguel/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ip/finn_design_dwc_1/synth/finn_design_dwc_1.v
output_final_miguel/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ip/finn_design_MatrixVectorActivation_1_0/sim/finn_design_MatrixVectorActivation_1_0.v
output_final_miguel/stitched_ip/finn_vivado_stitch_proj.gen/sources_1/bd/finn_design/ip/finn_design_MatrixVectorActivation_1_0/synth/finn_design_MatrixVectorActivati

In [16]:
# 14. Si falla HLS/IPGen, ejecutar esta celda para sacar el error real
# Busca errores dentro de FINN_build, incluyendo bitwidth, clang y failed.

build_root = "/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build"

if os.path.exists(build_root):
    print("Buscando errores en:", build_root)
    !grep -R "ERROR\|Error\|error\|CRITICAL\|failed\|Failed\|bitwidth" /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build | head -120
else:
    print("No existe:", build_root)

Buscando errores en: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build
/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_1_xccfasr7/hls_syn_MatrixVectorActivation_1.tcl:set_param hls.enable_hidden_option_error false
/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_1_xccfasr7/project_MatrixVectorActivation_1/sol1/.autopilot/xcd.log:Fri Jun 12 10:40:53 2026: XCD main fifo: do_read error code: Operation canceled
/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_1_xccfasr7/project_MatrixVectorActivation_1/sol1/.autopilot/db/top_MatrixVectorActivation_1.pp.0.cpp.clang.diag.xml:      <string>HLS pragma dump _XLX_SEP_ PragmaIsValid=1_XLX_SEP_ PragmaType=array_partition_XLX_SEP_ PragmaContext=_XLX_SEP_ PragmaFunction=TMRCheck_XLX_SEP_ PragmaOptions=variable=numerrors complete dim=0_XLX_SEP_</string>
/home/miguel_t

In [17]:
# 15. Alternativa recomendada: lanzar el build desde terminal/script en vez de Jupyter
# Esto crea un script build_miguel.py con la misma configuracion.
# Despues, en una terminal dentro de esta carpeta, ejecuta:
# python build_miguel.py > build_miguel_terminal.log 2>&1
# tail -f build_miguel_terminal.log

script = f"""
import os
import shutil

from finn.builder import build_dataflow as build
from finn.builder import build_dataflow_config as build_cfg

MODEL_TIDY_ONNX = "{MODEL_TIDY_ONNX}"
FINAL_OUTPUT_DIR = "{FINAL_OUTPUT_DIR}"
FOLDING_FILE = "{FOLDING_FILE}"
FPGA_PART = "{FPGA_PART}"

if os.path.exists(FINAL_OUTPUT_DIR):
    shutil.rmtree(FINAL_OUTPUT_DIR)

cfg = build.DataflowBuildConfig(
    output_dir=FINAL_OUTPUT_DIR,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
    ],
    folding_config_file=FOLDING_FILE,
    auto_fifo_depths=True,
    mvau_wwidth_max=36,
    target_fps=1000,
    synth_clk_period_ns=10.0,
    fpga_part=FPGA_PART
)

build.build_dataflow_cfg(MODEL_TIDY_ONNX, cfg)
print("Build finalizado. Revisa:", FINAL_OUTPUT_DIR)
"""

with open("build_miguel.py", "w") as f:
    f.write(script)

print("Creado build_miguel.py")
print("Comando recomendado en terminal:")
print("python build_miguel.py > build_miguel_terminal.log 2>&1")
print("tail -f build_miguel_terminal.log")

Creado build_miguel.py
Comando recomendado en terminal:
python build_miguel.py > build_miguel_terminal.log 2>&1
tail -f build_miguel_terminal.log
